In [4]:
# ============================================================
# BHARAT ENGLISH VALIDATION DATA PREPARATION
# Cell 1 — Imports
# ============================================================

import os
import re
import json
import math
import numpy as np
import pandas as pd

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# ============================================================
# Cell 2 — Connect Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully")

Mounted at /content/drive
Google Drive mounted successfully


In [5]:
# ============================================================
# Cell 3 — Find Bharat English Dataset in Google Drive
# ============================================================

import os

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file == "bharat_english_clean.jsonl":
            matches.append(os.path.join(root, file))

print("Files found:", len(matches))
print()

for path in matches:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print("File:", path)
    print(f"Size: {size_mb:.2f} MB")

Files found: 1

File: /content/drive/MyDrive/Bharat_LLM_Data/bharat_english_clean.jsonl
Size: 164.53 MB


In [6]:
# ============================================================
# Cell 4 — Load Bharat English Dataset
# ============================================================

input_file = "/content/drive/MyDrive/Bharat_LLM_Data/bharat_english_clean.jsonl"

df = pd.read_json(input_file, lines=True)

print("Bharat English dataset loaded successfully")
print("-" * 60)
print("File:", input_file)
print("Rows:", len(df))
print("Columns:", list(df.columns))

file_size_mb = os.path.getsize(input_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

Bharat English dataset loaded successfully
------------------------------------------------------------
File: /content/drive/MyDrive/Bharat_LLM_Data/bharat_english_clean.jsonl
Rows: 1231482
Columns: ['text']
File size: 164.53 MB


In [7]:
# ============================================================
# Cell 5 — Source Dataset Validation
# ============================================================

print("SOURCE DATASET VALIDATION")
print("=" * 60)

# 1. Missing values
missing_values = df["text"].isna().sum()

print("Missing values:", missing_values)

# 2. Empty records
empty_records = (
    df["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty records:", empty_records)

# 3. Duplicate records
duplicate_records = df["text"].duplicated().sum()

print("Duplicate records:", duplicate_records)

# 4. Minimum character length
character_lengths = df["text"].str.len()

print("\nCharacter statistics")
print("-" * 60)
print("Minimum:", character_lengths.min())
print("Maximum:", character_lengths.max())
print("Average:", round(character_lengths.mean(), 2))
print("Median :", character_lengths.median())

# 5. Dataset size
text_size_bytes = (
    df["text"]
    .fillna("")
    .apply(lambda x: len(x.encode("utf-8")))
    .sum()
)

text_size_mb = text_size_bytes / (1024 * 1024)

print("\nText size:", round(text_size_mb, 2), "MB")

# 6. Basic status
if (
    missing_values == 0
    and empty_records == 0
    and duplicate_records == 0
):
    print("\nSTATUS: SOURCE DATASET PASSED VALIDATION ✅")
else:
    print("\nSTATUS: SOURCE DATASET NEEDS REVIEW ⚠️")

SOURCE DATASET VALIDATION
Missing values: 0
Empty records: 0
Duplicate records: 0

Character statistics
------------------------------------------------------------
Minimum: 50
Maximum: 14820
Average: 127.72
Median : 108.0

Text size: 150.0 MB

STATUS: SOURCE DATASET PASSED VALIDATION ✅


In [8]:
# ============================================================
# Cell 6 — Create Separate Validation Dataset
# ============================================================

VALIDATION_TARGET_MB = 100
VALIDATION_TARGET_BYTES = VALIDATION_TARGET_MB * 1024 * 1024

# Use a fixed seed so the same split can be reproduced
RANDOM_SEED = 42

print("Creating validation dataset...")
print("-" * 60)
print(f"Target validation size: {VALIDATION_TARGET_MB} MB")
print(f"Random seed: {RANDOM_SEED}")

# Shuffle record order reproducibly
shuffled_df = df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

# Select records until approximately 100 MB of UTF-8 text
validation_records = []
validation_size = 0

for text in shuffled_df["text"]:

    text_size = len(text.encode("utf-8"))

    if validation_size + text_size > VALIDATION_TARGET_BYTES:
        break

    validation_records.append(text)
    validation_size += text_size

validation_df = pd.DataFrame({
    "text": validation_records
})

validation_size_mb = validation_size / (1024 * 1024)

print("\nValidation dataset created")
print("-" * 60)
print("Records:", len(validation_df))
print("Text size:", round(validation_size_mb, 2), "MB")

Creating validation dataset...
------------------------------------------------------------
Target validation size: 100 MB
Random seed: 42

Validation dataset created
------------------------------------------------------------
Records: 821026
Text size: 100.0 MB


In [9]:
# ============================================================
# Cell 7 — Check Validation / Training Separation
# ============================================================

validation_texts = set(validation_df["text"])

remaining_df = shuffled_df[
    ~shuffled_df["text"].isin(validation_texts)
].copy()

print("TRAIN / VALIDATION SEPARATION")
print("=" * 60)

print("Original records       :", len(df))
print("Validation records     :", len(validation_df))
print("Remaining records      :", len(remaining_df))

# Check overlap
overlap = set(validation_df["text"]).intersection(
    set(remaining_df["text"])
)

print("Validation overlap     :", len(overlap))

# Check duplicate records inside validation
validation_duplicates = validation_df["text"].duplicated().sum()

print("Validation duplicates  :", validation_duplicates)

if len(overlap) == 0 and validation_duplicates == 0:
    print("\nSTATUS: VALIDATION DATA IS PROPERLY SEPARATED ✅")
else:
    print("\nSTATUS: SEPARATION NEEDS REVIEW ⚠️")

TRAIN / VALIDATION SEPARATION
Original records       : 1231482
Validation records     : 821026
Remaining records      : 410456
Validation overlap     : 0
Validation duplicates  : 0

STATUS: VALIDATION DATA IS PROPERLY SEPARATED ✅


In [10]:
# ============================================================
# Cell 8 — Validation Dataset Quality Check
# ============================================================

print("VALIDATION DATA QUALITY CHECK")
print("=" * 60)

# Missing values
missing_values = validation_df["text"].isna().sum()

# Empty records
empty_records = (
    validation_df["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

# Duplicate records
duplicate_records = validation_df["text"].duplicated().sum()

# Character lengths
character_lengths = validation_df["text"].str.len()

# English / Latin character ratio
def latin_ratio(text):
    if not isinstance(text, str) or not text:
        return 0.0

    latin_count = sum(
        1
        for char in text
        if ('A' <= char <= 'Z') or
           ('a' <= char <= 'z')
    )

    return latin_count / len(text)


validation_df["latin_ratio"] = validation_df["text"].apply(
    latin_ratio
)

# Low-English records
low_english_records = (
    validation_df["latin_ratio"] < 0.50
).sum()

print("Missing values       :", missing_values)
print("Empty records        :", empty_records)
print("Duplicate records    :", duplicate_records)
print("Below 50 chars       :", (character_lengths < 50).sum())
print("Low-English records  :", low_english_records)

print("\nCharacter statistics")
print("-" * 60)
print("Minimum :", character_lengths.min())
print("Maximum :", character_lengths.max())
print("Average :", round(character_lengths.mean(), 2))
print("Median  :", character_lengths.median())

print("\nLatin-character ratio")
print("-" * 60)
print(validation_df["latin_ratio"].describe())

# Final status
if (
    missing_values == 0
    and empty_records == 0
    and duplicate_records == 0
    and (character_lengths < 50).sum() == 0
    and low_english_records == 0
):
    print("\nSTATUS: VALIDATION DATA PASSED QUALITY CHECK ✅")
else:
    print("\nSTATUS: VALIDATION DATA NEEDS REVIEW ⚠️")

VALIDATION DATA QUALITY CHECK
Missing values       : 0
Empty records        : 0
Duplicate records    : 0
Below 50 chars       : 0
Low-English records  : 0

Character statistics
------------------------------------------------------------
Minimum : 50
Maximum : 14820
Average : 127.72
Median  : 108.0

Latin-character ratio
------------------------------------------------------------
count    821026.000000
mean          0.805673
std           0.040881
min           0.500000
25%           0.789474
50%           0.814208
75%           0.831683
max           0.976471
Name: latin_ratio, dtype: float64

STATUS: VALIDATION DATA PASSED QUALITY CHECK ✅


In [11]:
# ============================================================
# Cell 9 — Prepare Final Validation Dataset
# ============================================================

# Keep only the actual text column
validation_final = validation_df[["text"]].copy()

print("FINAL VALIDATION DATASET")
print("=" * 60)
print("Records:", len(validation_final))

validation_text_bytes = (
    validation_final["text"]
    .apply(lambda x: len(x.encode("utf-8")))
    .sum()
)

validation_text_mb = validation_text_bytes / (1024 * 1024)

print("Text size:", round(validation_text_mb, 2), "MB")
print("Columns:", list(validation_final.columns))

print("\nSample records:")
for i, text in enumerate(validation_final["text"].head(5)):
    print(f"\n[{i}]")
    print(text[:500])

FINAL VALIDATION DATASET
Records: 821026
Text size: 100.0 MB
Columns: ['text']

Sample records:

[0]
The actress continues to keep her followers engaged by taking out time to post images for her fans on Instagram.

[1]
Members from across the Jammu region were present at the meeting.

[2]
"So he understands what they're going through, but despises their destructive force."""

[3]
We are not working for any party, we are supplying strategic products to IAF and Indian Govt.

[4]
Ms.Nivruti Rai, Country Head, Intel India, and Dr.Aloknath De, Chief Technology Officer, Samsung R&D, will address a session on collaboration and data driven research and decision making


In [12]:
# ============================================================
# Cell 10 — Export Validation Dataset to Google Drive
# ============================================================

output_file = (
    "/content/drive/MyDrive/"
    "Bharat_LLM_Data/"
    "bharat_english_validation.jsonl"
)

validation_final.to_json(
    output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Validation dataset exported successfully")
print("=" * 60)
print("File:", output_file)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)

print(f"File size: {file_size_mb:.2f} MB")
print("Records:", len(validation_final))

Validation dataset exported successfully
File: /content/drive/MyDrive/Bharat_LLM_Data/bharat_english_validation.jsonl
File size: 109.68 MB
Records: 821026


In [13]:
# ============================================================
# Cell 11 — Final Export Verification
# ============================================================

print("FINAL EXPORT VERIFICATION")
print("=" * 60)

# Check file exists
print("File exists:", os.path.exists(output_file))

# File size
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

# Reload the exported JSONL
verified_validation = pd.read_json(
    output_file,
    lines=True
)

print("Rows:", len(verified_validation))
print("Columns:", list(verified_validation.columns))

# Validation checks
missing = verified_validation["text"].isna().sum()

empty = (
    verified_validation["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

duplicates = verified_validation["text"].duplicated().sum()

print("\nQuality checks")
print("-" * 60)
print("Missing values:", missing)
print("Empty records :", empty)
print("Duplicates    :", duplicates)

if (
    len(verified_validation) == len(validation_final)
    and missing == 0
    and empty == 0
    and duplicates == 0
):
    print("\nSTATUS: VALIDATION DATASET READY ✅")
else:
    print("\nSTATUS: REVIEW REQUIRED ⚠️")

FINAL EXPORT VERIFICATION
File exists: True
File size: 109.68 MB
Rows: 821026
Columns: ['text']

Quality checks
------------------------------------------------------------
Missing values: 0
Empty records : 0
Duplicates    : 0

STATUS: VALIDATION DATASET READY ✅


In [14]:
# ============================================================
# Cell 12 — Load Bharat Hindi Dataset
# ============================================================

hindi_input_file = (
    "/content/drive/MyDrive/"
    "Bharat_LLM_Data/"
    "bharat_hindi_clean.jsonl"
)

if not os.path.exists(hindi_input_file):
    raise FileNotFoundError(
        f"Hindi dataset not found:\n{hindi_input_file}"
    )

hindi_file_size_mb = (
    os.path.getsize(hindi_input_file)
    / (1024 * 1024)
)

print("Bharat Hindi dataset found")
print("-" * 60)
print("File:", hindi_input_file)
print(f"File size: {hindi_file_size_mb:.2f} MB")

hindi_df = pd.read_json(
    hindi_input_file,
    lines=True
)

print("\nHindi dataset loaded successfully")
print("-" * 60)
print("Rows:", len(hindi_df))
print("Columns:", list(hindi_df.columns))

Bharat Hindi dataset found
------------------------------------------------------------
File: /content/drive/MyDrive/Bharat_LLM_Data/bharat_hindi_clean.jsonl
File size: 152.10 MB

Hindi dataset loaded successfully
------------------------------------------------------------
Rows: 182476
Columns: ['text']


In [15]:
# ============================================================
# Cell 13 — Hindi Source Dataset Validation
# ============================================================

print("HINDI SOURCE DATASET VALIDATION")
print("=" * 60)

# 1. Missing values
missing_values = hindi_df["text"].isna().sum()

# 2. Empty records
empty_records = (
    hindi_df["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

# 3. Duplicate records
duplicate_records = hindi_df["text"].duplicated().sum()

# 4. Character statistics
character_lengths = hindi_df["text"].str.len()

print("Missing values:", missing_values)
print("Empty records :", empty_records)
print("Duplicate records:", duplicate_records)

print("\nCharacter statistics")
print("-" * 60)
print("Minimum:", character_lengths.min())
print("Maximum:", character_lengths.max())
print("Average:", round(character_lengths.mean(), 2))
print("Median :", character_lengths.median())

# 5. Calculate Devanagari ratio
def devanagari_ratio(text):
    if not isinstance(text, str) or not text:
        return 0.0

    devanagari_count = sum(
        1
        for char in text
        if "\u0900" <= char <= "\u097F"
    )

    return devanagari_count / len(text)


hindi_df["devanagari_ratio"] = hindi_df["text"].apply(
    devanagari_ratio
)

print("\nDevanagari ratio statistics")
print("-" * 60)
print(hindi_df["devanagari_ratio"].describe())

# 6. Check Hindi quality
low_devanagari_records = (
    hindi_df["devanagari_ratio"] < 0.30
).sum()

print("\nRecords below 30% Devanagari:",
      low_devanagari_records)

# 7. Final status
if (
    missing_values == 0
    and empty_records == 0
    and duplicate_records == 0
    and low_devanagari_records == 0
):
    print("\nSTATUS: HINDI SOURCE DATASET PASSED VALIDATION ✅")
else:
    print("\nSTATUS: HINDI SOURCE DATASET NEEDS REVIEW ⚠️")

HINDI SOURCE DATASET VALIDATION
Missing values: 0
Empty records : 0
Duplicate records: 0

Character statistics
------------------------------------------------------------
Minimum: 50
Maximum: 100259
Average: 336.32
Median : 254.0

Devanagari ratio statistics
------------------------------------------------------------
count    182476.000000
mean          0.783841
std           0.035444
min           0.315789
25%           0.771679
50%           0.789474
75%           0.803922
max           1.000000
Name: devanagari_ratio, dtype: float64

Records below 30% Devanagari: 0

STATUS: HINDI SOURCE DATASET PASSED VALIDATION ✅


In [16]:
# ============================================================
# Cell 14 — Create Hindi Validation Dataset
# ============================================================

HINDI_VALIDATION_TARGET_MB = 100
HINDI_VALIDATION_TARGET_BYTES = (
    HINDI_VALIDATION_TARGET_MB * 1024 * 1024
)

RANDOM_SEED = 42

print("Creating Hindi validation dataset...")
print("-" * 60)
print(f"Target validation size: {HINDI_VALIDATION_TARGET_MB} MB")
print(f"Random seed: {RANDOM_SEED}")

# Shuffle reproducibly
hindi_shuffled_df = hindi_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

# Select approximately 100 MB of text
hindi_validation_records = []
hindi_validation_size = 0

for text in hindi_shuffled_df["text"]:

    text_size = len(text.encode("utf-8"))

    if (
        hindi_validation_size + text_size
        > HINDI_VALIDATION_TARGET_BYTES
    ):
        break

    hindi_validation_records.append(text)
    hindi_validation_size += text_size

hindi_validation_df = pd.DataFrame({
    "text": hindi_validation_records
})

hindi_validation_size_mb = (
    hindi_validation_size / (1024 * 1024)
)

print("\nHindi validation dataset created")
print("-" * 60)
print("Records:", len(hindi_validation_df))
print(
    "Text size:",
    round(hindi_validation_size_mb, 2),
    "MB"
)

Creating Hindi validation dataset...
------------------------------------------------------------
Target validation size: 100 MB
Random seed: 42

Hindi validation dataset created
------------------------------------------------------------
Records: 121849
Text size: 100.0 MB


In [17]:
# ============================================================
# Cell 15 — Check Hindi Train / Validation Separation
# ============================================================

hindi_validation_texts = set(
    hindi_validation_df["text"]
)

hindi_remaining_df = hindi_shuffled_df[
    ~hindi_shuffled_df["text"].isin(
        hindi_validation_texts
    )
].copy()

print("HINDI TRAIN / VALIDATION SEPARATION")
print("=" * 60)

print(
    "Original records       :",
    len(hindi_df)
)

print(
    "Validation records     :",
    len(hindi_validation_df)
)

print(
    "Remaining records      :",
    len(hindi_remaining_df)
)

# Check overlap
hindi_overlap = (
    set(hindi_validation_df["text"])
    .intersection(set(hindi_remaining_df["text"]))
)

# Check duplicates inside validation
hindi_validation_duplicates = (
    hindi_validation_df["text"].duplicated().sum()
)

print(
    "Validation overlap     :",
    len(hindi_overlap)
)

print(
    "Validation duplicates  :",
    hindi_validation_duplicates
)

if (
    len(hindi_overlap) == 0
    and hindi_validation_duplicates == 0
):
    print(
        "\nSTATUS: HINDI VALIDATION DATA "
        "IS PROPERLY SEPARATED ✅"
    )
else:
    print(
        "\nSTATUS: SEPARATION NEEDS REVIEW ⚠️"
    )

HINDI TRAIN / VALIDATION SEPARATION
Original records       : 182476
Validation records     : 121849
Remaining records      : 60627
Validation overlap     : 0
Validation duplicates  : 0

STATUS: HINDI VALIDATION DATA IS PROPERLY SEPARATED ✅


In [18]:
# ============================================================
# Cell 16 — Hindi Validation Dataset Quality Check
# ============================================================

print("HINDI VALIDATION DATA QUALITY CHECK")
print("=" * 60)

# Missing values
missing_values = hindi_validation_df["text"].isna().sum()

# Empty records
empty_records = (
    hindi_validation_df["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

# Duplicate records
duplicate_records = (
    hindi_validation_df["text"].duplicated().sum()
)

# Character statistics
character_lengths = hindi_validation_df["text"].str.len()

# Devanagari ratio
def devanagari_ratio(text):
    if not isinstance(text, str) or not text:
        return 0.0

    devanagari_count = sum(
        1
        for char in text
        if "\u0900" <= char <= "\u097F"
    )

    return devanagari_count / len(text)


hindi_validation_df["devanagari_ratio"] = (
    hindi_validation_df["text"].apply(
        devanagari_ratio
    )
)

# Records with less than 30% Devanagari
low_devanagari_records = (
    hindi_validation_df["devanagari_ratio"] < 0.30
).sum()

# Records below minimum character length
short_records = (
    character_lengths < 50
).sum()

print("Missing values        :", missing_values)
print("Empty records         :", empty_records)
print("Duplicate records     :", duplicate_records)
print("Below 50 chars        :", short_records)
print(
    "Low-Devanagari records:",
    low_devanagari_records
)

print("\nCharacter statistics")
print("-" * 60)
print("Minimum :", character_lengths.min())
print("Maximum :", character_lengths.max())
print("Average :", round(character_lengths.mean(), 2))
print("Median  :", character_lengths.median())

print("\nDevanagari ratio statistics")
print("-" * 60)
print(hindi_validation_df["devanagari_ratio"].describe())

if (
    missing_values == 0
    and empty_records == 0
    and duplicate_records == 0
    and short_records == 0
    and low_devanagari_records == 0
):
    print(
        "\nSTATUS: HINDI VALIDATION DATA "
        "PASSED QUALITY CHECK ✅"
    )
else:
    print(
        "\nSTATUS: HINDI VALIDATION DATA "
        "NEEDS REVIEW ⚠️"
    )

HINDI VALIDATION DATA QUALITY CHECK
Missing values        : 0
Empty records         : 0
Duplicate records     : 0
Below 50 chars        : 0
Low-Devanagari records: 0

Character statistics
------------------------------------------------------------
Minimum : 50
Maximum : 57013
Average : 335.77
Median  : 255.0

Devanagari ratio statistics
------------------------------------------------------------
count    121849.000000
mean          0.783730
std           0.035424
min           0.315789
25%           0.771639
50%           0.789474
75%           0.803874
max           1.000000
Name: devanagari_ratio, dtype: float64

STATUS: HINDI VALIDATION DATA PASSED QUALITY CHECK ✅


In [19]:
# ============================================================
# Cell 17 — Prepare Final Hindi Validation Dataset
# ============================================================

hindi_validation_final = hindi_validation_df[["text"]].copy()

print("FINAL HINDI VALIDATION DATASET")
print("=" * 60)
print("Records:", len(hindi_validation_final))
print("Columns:", list(hindi_validation_final.columns))

hindi_validation_text_bytes = (
    hindi_validation_final["text"]
    .apply(lambda x: len(x.encode("utf-8")))
    .sum()
)

hindi_validation_text_mb = (
    hindi_validation_text_bytes / (1024 * 1024)
)

print(
    "Text size:",
    round(hindi_validation_text_mb, 2),
    "MB"
)

print("\nSample Hindi records:")

for i, text in enumerate(
    hindi_validation_final["text"].head(5)
):
    print(f"\n[{i}]")
    print(text[:500])

FINAL HINDI VALIDATION DATASET
Records: 121849
Columns: ['text']
Text size: 100.0 MB

Sample Hindi records:

[0]
शनिवार को डिंपल यादव गोंडा के धानेपुर व वजीरगंज में जनसभा को संबोधित करने के बाद मंच से उतरीं तो पत्रकारों ने उनसे सवाल किया कि इलाहाबाद के हलिया में जब लोग सेल्फी के लिए आगे बढ़े थे तो आपने ने कहा था कि इसकी शिकायत भइया से करूंगी। इस घटना पर केंद्रीय मंत्री स्मृति ईरानी ने कहा है कि भाजपा की सरकार बनने दीजिए डिंपल को सुरक्षा मुहैया कराई जाएगी। उनके इस बयान पर आपका क्या कहना है?

[1]
भुवनेश्वर की फ्लाइट में जा रहे पैसेंजरों ने कहा कि अगर जज वीवीआईपी स्टेटस का मिसयूज कर रहे हैं, तो वे भोपाल की फ्लाइट काे नहीं उड़ने देंगे।

[2]
प्रो-एक्टिव गवर्नेंस के तहत पहली बार ऐसा हो रहा है कि नए बुजुर्गों को जनवरी माह के अंत में 600 रुपये प्रतिमाह पेंशन मिलना शुरु हो जाएगी। सभी 131 ग्राम पंचायतों में यह काम अंतिम दौर में है। इसके साथ ही पात्रता पर्ची भी बनाने की तैयारियां कर ली गई है। जो जनवरी माह से उन्हें दे दी जाएगी।

[3]
कांग्रेस ने आरोप लगाया है कि दिसंबर 2020 में ईडी के एक पत्र के ब

In [20]:
# ============================================================
# Cell 18 — Export Hindi Validation Dataset to Google Drive
# ============================================================

hindi_output_file = (
    "/content/drive/MyDrive/"
    "Bharat_LLM_Data/"
    "bharat_hindi_validation.jsonl"
)

hindi_validation_final.to_json(
    hindi_output_file,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Hindi validation dataset exported successfully")
print("=" * 60)
print("File:", hindi_output_file)

hindi_file_size_mb = (
    os.path.getsize(hindi_output_file)
    / (1024 * 1024)
)

print(
    f"File size: {hindi_file_size_mb:.2f} MB"
)
print(
    "Records:",
    len(hindi_validation_final)
)

Hindi validation dataset exported successfully
File: /content/drive/MyDrive/Bharat_LLM_Data/bharat_hindi_validation.jsonl
File size: 101.40 MB
Records: 121849


In [21]:
# ============================================================
# Cell 19 — Final Hindi Export Verification
# ============================================================

print("FINAL HINDI EXPORT VERIFICATION")
print("=" * 60)

print(
    "File exists:",
    os.path.exists(hindi_output_file)
)

hindi_file_size_mb = (
    os.path.getsize(hindi_output_file)
    / (1024 * 1024)
)

print(
    f"File size: {hindi_file_size_mb:.2f} MB"
)

verified_hindi = pd.read_json(
    hindi_output_file,
    lines=True
)

print("Rows:", len(verified_hindi))
print("Columns:", list(verified_hindi.columns))

missing = verified_hindi["text"].isna().sum()

empty = (
    verified_hindi["text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

duplicates = verified_hindi["text"].duplicated().sum()

print("\nQuality checks")
print("-" * 60)
print("Missing values:", missing)
print("Empty records :", empty)
print("Duplicates    :", duplicates)

if (
    len(verified_hindi) == len(hindi_validation_final)
    and missing == 0
    and empty == 0
    and duplicates == 0
):
    print(
        "\nSTATUS: HINDI VALIDATION DATASET READY ✅"
    )
else:
    print(
        "\nSTATUS: REVIEW REQUIRED ⚠️"
    )

FINAL HINDI EXPORT VERIFICATION
File exists: True
File size: 101.40 MB
Rows: 121849
Columns: ['text']

Quality checks
------------------------------------------------------------
Missing values: 0
Empty records : 0
Duplicates    : 0

STATUS: HINDI VALIDATION DATASET READY ✅
